<a href="https://colab.research.google.com/github/SNK005/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SNK005/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type:**

 This is a classification problem at its core — predicting the probability that a page falls into the "declining" class — used to produce a ranking/scoring output. The classifier's probability output is not consumed as a hard yes/no; it is used to order pages from highest to lowest priority, so the final deliverable is a ranked queue, not a binary label.

This matches the ML-02 pipeline pattern, where a classifier such as a decision tree or logistic regression produces probabilities that are then sorted into a review queue. It is not a clustering task because the target class is predefined, and it is not a pure pairwise ranking task because pages are scored independently and then sorted rather than being trained on explicit "A should rank above B" comparisons.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target:** **`is_declining_label`**

The target is `is_declining_label`, a yes/no label based on whether `trend_direction == "down"` in the current 90-day period. The model estimates how likely each page is to be declining. This probability becomes the priority score used to rank pages for review.

This target describes whether a page is declining **right now**, not whether it will decline in the future. It also does not tell us whether refreshing the page will improve its performance. It is therefore a proxy target, rather than the strongest possible target. A stronger version could use data from the previous 90 days to predict whether a page will decline in the **next 30 days**. This would help identify pages that may become problems before they actually decline.

Because `trend_direction` and `trend_pct` are used to create the target, they must not be used as model inputs. Otherwise, the model would be given information that directly reveals the answer, which is called **data leakage**.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Success metric: Precision@50** (or whatever K matches realistic reviewer capacity), as the primary metric. This matches how the output is actually used: a reviewer works through a fixed-size queue, so what matters is how many of the top-ranked pages are actually relevant, rather than overall accuracy across all 30,000 pages. For example, if 40 of the top 50 ranked pages are actually declining, Precision@50 would be 0.80.

I will also track **recall as a secondary metric** because, as noted in ML-02, missing a genuinely declining page can be costly. Recall helps check whether the model is failing to identify too many of the pages that are actually declining, even if Precision@50 looks good.

I will not use plain accuracy as the main metric because roughly 54% of pages are already labeled as declining in the starter data. An accuracy score could therefore look reasonable without telling us whether the model is actually useful for prioritizing the limited review queue.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

 **Unit of analysis**

One row represents one content page/content item (`content_id`) with its trailing 90-day performance and page-level characteristics.

The dataframe shows that each `content_id` is treated as one observation. For example, `is_declining_label = 1` when `trend_direction` is `down`, while `is_declining_label = 0` when it is not.

The model would ultimately give each page a score, which can then be used to rank pages for the content review queue.

In [2]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/SNK005/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv")

# Same qualifying filter as ML-02
df_valid = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates(subset="content_id")

# Show the unit of analysis: one row = one page
df_valid["is_declining_label"] = (df_valid["trend_direction"] == "down").astype(int)

cols_to_show = ["content_id", "impressions_90d", "content_age_days", "trend_direction", "is_declining_label"]
df_valid[cols_to_show].head(10)

,content_id,impressions_90d,content_age_days,trend_direction,is_declining_label
0,content_304f48230142,3803,187,down,1
1,content_a1fb4e703a9e,15320,445,down,1
2,content_9aa793d4d895,12581,141,down,1
3,content_331d6c4de07b,11751,463,stable,0
4,content_d99b7a2d90ca,19140,263,down,1
5,content_d4084a4bc775,3970,147,down,1
6,content_9a34b442b552,20,90,down,1
7,content_a63219c6e95a,1724,445,stable,0
8,content_5e6c160719bc,32574,90,down,1
9,content_c27558df2b0c,1240,257,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*



A fair fixed rule has to use only information available before prediction, and cannot reference the column the target was built from. For example: "review pages older than 180 days with impressions_90d >= 500" — this does not leak `trend_direction` into the decision, unlike a rule that directly checks the trend itself.

I have not yet measured this specific rule's Precision@50 on my data, so I cannot currently claim that my model beats it — that comparison still needs to be run, not assumed.

The lane guide's starter results provide supporting evidence from the same problem space: a leakage-safe baseline-rules approach reached Precision@50 = 0.240, while a random forest reached 0.740. However, this is the guide's comparison, not my own result.

I therefore need to confirm on my own data whether an ML model actually adds value over a fair fixed-rule or naive baseline, with both approaches evaluated in the same way.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.